# Exercise: Implementing a Mock Executor

Welcome to the programming assignment! In this notebook, you'll get hands-on experience with a key component of a modern LLM serving system: the **Executor**.

The Executor acts as the bridge between incoming user requests and the powerful model running on a computational device (like a GPU). It's responsible for batching requests, preparing data in a format the model understands, executing the model, and processing the results.

In this exercise, you will:
*   Implement `prepare_model_inputs` to extract the necessary data for the model from a batch of requests.
*   Implement `process_model_outputs` to package the model's raw output back into a structured format.
*   See how these functions work together inside a `MockExecutor` to run a complete inference step.

Let's get started!

In [ ]:
import dataclasses
from typing import List, Dict

# --- Helper Code (Do not modify) ---

@dataclasses.dataclass
class Request:
    """Represents a single user request to the LLM."""
    request_id: str
    input_ids: List[int]
    # In a real system, this would map logical data blocks to physical memory blocks.
    # We include it here to make our simulation more realistic.
    kv_cache_mapping: Dict[int, int]

@dataclasses.dataclass
class MockOutput:
    """Represents the output for a single request after one step of generation."""
    request_id: str
    next_token_id: int

VOCAB_SIZE = 32000

class MockModel:
    """
    A fake model that simulates running a forward pass on a computational device.
    It takes a batch of the last token IDs and "generates" the next token ID
    for each request in a deterministic way.
    """
    def __init__(self, vocab_size: int):
        self.vocab_size = vocab_size

    def forward(self, last_token_ids: List[int]) -> List[int]:
        """
        Simulates a forward pass. The logic is deterministic for testing purposes.
        """
        # A simple, deterministic transformation for our simulation.
        return [(token_id * 101) % self.vocab_size for token_id in last_token_ids]

print("Setup complete. Helper classes and MockModel are defined.")

### Exercise 1: Prepare Model Inputs

The `MockModel` doesn't understand our `Request` objects. It expects a simple list of token IDs to process.

For auto-regressive decoding (where the model predicts the next word based on the previous ones), the model only needs the **very last token** of each sequence in the batch to predict the next one. Your task is to implement a function that extracts this last token ID from each request in the batch.

**Hint:** Python's negative indexing (e.g., `my_list[-1]`) is very useful for getting the last item from a list.

In [ ]:
def prepare_model_inputs(requests: List[Request]) -> List[int]:
    """
    Extracts the last token ID from each request in a batch.

    Args:
        requests: A list of Request objects.

    Returns:
        A list of integers, where each integer is the last token ID
        from the corresponding request.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

In [ ]:
# --- Test Cell for Exercise 1 ---

# 1. Create a batch of sample requests
sample_requests_1 = [
    Request(request_id="req_1", input_ids=[10, 20, 30], kv_cache_mapping={}),
    Request(request_id="req_2", input_ids=[1, 2, 3, 4, 5], kv_cache_mapping={}),
    Request(request_id="req_3", input_ids=[999], kv_cache_mapping={}),
]

# 2. Call your function
prepared_inputs = prepare_model_inputs(sample_requests_1)

# 3. Check the result
print(f"Input requests: {sample_requests_1}")
print(f"Prepared model inputs: {prepared_inputs}")

expected_inputs = [30, 5, 999]
assert prepared_inputs == expected_inputs, f"Test Failed! Got {prepared_inputs}, but expected {expected_inputs}"

# Test with an empty list
assert prepare_model_inputs([]) == [], "Test Failed! Should handle an empty list of requests."

print("\n✅ All tests passed for prepare_model_inputs!")

### Exercise 2: Process Model Outputs

Fantastic! Now for the other side of the process.

After the model runs, it gives us a simple list of the next token IDs—one for each request in the batch. We need to associate these new tokens back with their original requests. Your function will take the original list of requests and the new token IDs and create a `MockOutput` object for each one, which includes the `request_id`.

**Hint:** The `zip()` function is perfect for iterating over two lists at the same time. You'll want to combine each `request` with its corresponding `next_token_id`.

In [ ]:
def process_model_outputs(requests: List[Request], next_token_ids: List[int]) -> List[MockOutput]:
    """
    Packages the model's output token IDs into MockOutput objects.

    Args:
        requests: The original list of Request objects in the batch.
        next_token_ids: The list of generated token IDs from the model.

    Returns:
        A list of MockOutput objects, associating each request with its
        generated next token.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

In [ ]:
# --- Test Cell for Exercise 2 ---

# 1. Create a batch of sample requests (can reuse from the previous test)
sample_requests_2 = [
    Request(request_id="req_A", input_ids=[10], kv_cache_mapping={}),
    Request(request_id="req_B", input_ids=[20, 30], kv_cache_mapping={}),
]

# 2. Create a list of mock generated IDs
generated_ids = [1234, 5678]

# 3. Call your function
processed_outputs = process_model_outputs(sample_requests_2, generated_ids)

# 4. Check the result
print(f"Original requests: {sample_requests_2}")
print(f"Generated token IDs: {generated_ids}")
print(f"Processed outputs: {processed_outputs}")

expected_outputs = [
    MockOutput(request_id='req_A', next_token_id=1234),
    MockOutput(request_id='req_B', next_token_id=5678)
]
assert processed_outputs == expected_outputs, f"Test Failed! Got {processed_outputs}, but expected {expected_outputs}"

# Test with an empty list
assert process_model_outputs([], []) == [], "Test Failed! Should handle empty lists."

print("\n✅ All tests passed for process_model_outputs!")

### Putting It All Together: The `MockExecutor`

Excellent work! You've built the two key data transformation steps. Now, let's see how they fit into a class that represents the executor.

The `MockExecutor` class below orchestrates the entire process for a single step. Its `execute_step` method:
1.  Calls your `prepare_model_inputs` function.
2.  Passes the prepared inputs to the `MockModel`.
3.  Calls your `process_model_outputs` function to package the results.

Run the cell below to see your functions in action and complete the final integration test!

In [ ]:
class MockExecutor:
    """
    Orchestrates a single step of the LLM inference process.
    """
    def __init__(self, model: MockModel):
        self.model = model

    def execute_step(self, requests: List[Request]) -> List[MockOutput]:
        """
        Runs one forward pass for a batch of requests.
        """
        print("--- Executing one step ---")

        # 1. Prepare data for the model using your first function.
        print("1. Preparing model inputs...")
        last_token_ids = prepare_model_inputs(requests)
        print(f"   - Prepared inputs (last token IDs): {last_token_ids}")

        # 2. Run the model's forward pass.
        print("2. Executing model forward pass...")
        next_token_ids = self.model.forward(last_token_ids)
        print(f"   - Model generated IDs: {next_token_ids}")

        # 3. Process the model's output using your second function.
        print("3. Processing model outputs...")
        outputs = process_model_outputs(requests, next_token_ids)
        print(f"   - Final structured outputs: {outputs}")

        print("--- Step execution complete ---")
        return outputs

# --- Integration Test ---

# Use the same requests from the first test
test_requests = [
    Request(request_id="req_1", input_ids=[10, 20, 30], kv_cache_mapping={}),
    Request(request_id="req_2", input_ids=[1, 2, 3, 4, 5], kv_cache_mapping={}),
    Request(request_id="req_3", input_ids=[999], kv_cache_mapping={}),
]

# Instantiate our components
mock_model = MockModel(vocab_size=VOCAB_SIZE)
executor = MockExecutor(model=mock_model)

# Run the full step
final_outputs = executor.execute_step(test_requests)

# Calculate the expected outputs based on the MockModel's known behavior
# last_tokens = [30, 5, 999]
# next_tokens = [(30*101)%32000, (5*101)%32000, (999*101)%32000] -> [3030, 505, 100899 % 32000 = 4899]
expected_final_outputs = [
    MockOutput(request_id='req_1', next_token_id=3030),
    MockOutput(request_id='req_2', next_token_id=505),
    MockOutput(request_id='req_3', next_token_id=4899),
]

assert final_outputs == expected_final_outputs, f"Integration Test Failed! Got {final_outputs}, but expected {expected_final_outputs}"

print("\n🎉 Congratulations! All tests passed and the MockExecutor works as expected!")
print("You have successfully implemented the core data flow of an LLM executor.")